<a href="https://colab.research.google.com/github/kligeys/AI-Projects/blob/main/single_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import json
import os
import requests
from google.colab import userdata
from openai import OpenAI

In [43]:
# 1. Initialize API Keys
# It's best practice to use environment variables, but you can hardcode them for testing.
# 1. Retrieve the API key safely from Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')
weather_api_key = userdata.get('OPENWEATHER_API_KEY')
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", api_key)
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY", weather_api_key)

In [44]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [53]:
import json
import os
import requests
from google.colab import userdata
from openai import OpenAI

# 2. Define the Weather Tool Function
def get_weather_forecast(location):
    """Fetches real-time weather information using the OpenWeatherMap API."""
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": weather_api_key,
        "units": "metric",  # Use 'imperial' for Fahrenheit
    }

    try:
        response = requests.get(base_url, params=params)
        print("the response code is", response.status_code)
        if response.status_code == 200:
            data = response.json()
            weather_desc = data["weather"][0]["description"]
            temp = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            return json.dumps(
                {
                    "location": location,
                    "temperature": f"{temp}°C",
                    "condition": weather_desc,
                    "humidity": f"{humidity}%",
                },
                ensure_ascii=False # Add this to prevent Unicode escaping
            )
        else:
            return json.dumps(
                {"error": f"Could not find weather for '{location}'. Status Code: {response.status_code}"},
                ensure_ascii=False # Add this to prevent Unicode escaping
            )
    except Exception as e:
        return json.dumps({"error": str(e)}, ensure_ascii=False)

In [54]:
# 3. Define the Tool Specification for OpenAI
# This tells ChatGPT what the function does and what arguments it accepts.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_forecast",
            "description": "Get the current weather or forecast for a given city/location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state/country, e.g., San Francisco, CA or Paris, France",
                    }
                },
                "required": ["location"],
            },
        },
    }
]

In [55]:
# 4. Agent Execution Loop
def run_weather_agent(user_prompt: str):
    print(f"User: {user_prompt}\n")

    # Step A: Send the initial conversation history to ChatGPT
    messages = [
        {
            "role": "system",
            "content": "You are a helpful weather assistant. Use the provided tools to lookup actual weather data before answering.",
        },
        {"role": "user", "content": user_prompt},
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=tools
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Step B: Check if the model wants to call a tool
    if tool_calls:
        # Append the assistant's request to call a tool to the conversation history
        messages.append(response_message)

        # Process each tool call requested by the model
        for tool_call in tool_calls:
            if tool_call.function.name == "get_weather_forecast":
                # Parse the arguments generated by ChatGPT
                function_args = json.loads(tool_call.function.arguments)
                location_argument = function_args.get("location")

                print(
                    f"🤖 [Agent Action] Executing local tool 'get_weather_forecast' for: {location_argument}..."
                )

                # Execute the actual Python tool function
                tool_output = get_weather_forecast(location_argument)
                print(f"🤖 [Tool Output] {tool_output}") # Added this line to print tool output

                # Send the tool result back into the conversation history
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": "get_weather_forecast",
                        "content": tool_output,
                    }
                )

        # Step C: Get the final synthesis from ChatGPT with the data we provided
        final_response = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages
        )
        print(f"\nAgent: {final_response.choices[0].message.content}")
    else:
        # If the query didn't need a weather lookup (e.g., "Hello!"), just print the response
        print(f"Agent: {response_message.content}")

In [65]:
# Run the agent
if __name__ == "__main__":
    query = "What is the weather like in California right now? Should I take any precautions?"
    run_weather_agent(query)

User: What is the weather like in California right now? Should I take any precautions?

🤖 [Agent Action] Executing local tool 'get_weather_forecast' for: California...
the response code is 200
🤖 [Tool Output] {"location": "California", "temperature": "14.71°C", "condition": "overcast clouds", "humidity": "93%"}

Agent: The current weather in California is 14.71°C with overcast clouds and a humidity level of 93%. Given the high humidity and overcast conditions, it may feel cooler, and there's a chance of precipitation. 

**Precautions to consider:**
- Dress in layers to stay comfortable.
- If you're planning to be outdoors, be aware of the possibility of rain, so carrying an umbrella or raincoat could be wise.
- Stay hydrated, even in cooler conditions, as humidity can be deceptive.

If you need local updates or specific precautions for a particular area in California, let me know!
